# Fase 1 — Relevamiento del dominio, supuestos y requerimientos

**Proyecto:** Análisis de la capacidad de respuesta sanitaria ante la incidencia histórica de dengue en el Perú (2000-2024)

**Pregunta de negocio:** ¿los distritos (ubigeos) con más casos históricos de dengue cuentan con infraestructura de salud suficiente (cantidad y categoría de IPRESS) para soportar futuros brotes?

## 1. Datasets seleccionados

| Dataset | Fuente y URL | Última modificación en la fuente | Fecha de descarga | 1 fila = |
|---|---|---|---|---|
| `datos_abiertos_vigilancia_dengue_2000_2024.csv` | CDC Perú / MINSA. https://www.datosabiertos.gob.pe/dataset/vigilancia-epidemiol%C3%B3gica-de-dengue | 17/12/2025 | `[completar]` | 1 caso notificado |
| `IPRESS.csv` | SUSALUD / MINSA. https://www.datosabiertos.gob.pe/dataset/minsa-ipress | 07/08/2025 | `[completar]` | 1 establecimiento de salud |

**Relación entre ambos:** solo geográfica, por `ubigeo` (el dengue no trae código de establecimiento).
**Volumen, esquema original y problemas de calidad con conteo:** se generan con código en las celdas siguientes.

## 2. Supuestos declarados

| # | Supuesto | Justificación técnica | Impacto en el diseño |
|---|---|---|---|
| S1 | Las filas idénticas de dengue son pacientes distintos y se conservan | No existe ID de paciente; eliminarlas subestimaría la morbilidad | La tabla de hechos mantiene granularidad de caso; el ETL no deduplica |
| S2 | Ubigeo vacío o inválido se asigna a un miembro "Desconocido" (`000000`) | Descartar registros subestimaría el total nacional | `Dim_Ubigeo` incluye la fila desconocido; el ETL valida contra el catálogo |
| S3 | IPRESS es una foto actual: se compara incidencia histórica vs. infraestructura de hoy | El archivo no trae historial ni fecha de cierre | El KPI mide cobertura actual; no se reconstruye capacidad por año |
| S4 | Se conservan todas las IPRESS; la capacidad se define solo con establecimientos con internamiento o categoría ≥ I-3 *(criterio por confirmar)* | La mayoría son consultorios privados que no aportan capacidad en un brote | La dimensión IPRESS lleva una bandera `es_capacidad` y el KPI filtra por ella |

## 3. Requerimientos funcionales

| # | Requerimiento | Criterio de éxito verificable |
|---|---|---|
| RF1 | Ingesta automatizada a SQL Server con procedimientos almacenados (TRY/CATCH) | Conteo origen = conteo destino; toda ejecución queda en el log |
| RF2 | Seguridad conforme a la Ley N.° 29733: 3 roles y auditoría | Cada rol solo ejecuta lo permitido (pruebas documentadas); los triggers registran INSERT/UPDATE/DELETE |
| RF3 | Dashboard de capacidad de respuesta | ≥ 3 KPIs con drill-down región → distrito; sus cifras coinciden con las del Data Warehouse |

In [1]:
# ==============================================================================
# Fase 2: CARGA VERIFICADA Y DIAGNÓSTICO DE CALIDAD
# ==============================================================================
import pandas as pd
import re, unicodedata
from IPython.display import display

F_IPRESS = 'IPRESS.csv'
F_DENGUE = 'datos_abiertos_vigilancia_dengue_2000_2024.csv'

# --- 1. Carga (todo como texto para no perder ceros a la izquierda) ---
df_ipress = pd.read_csv(F_IPRESS, encoding='latin-1', dtype=str,
                        keep_default_na=False, on_bad_lines='warn')
df_dengue = pd.read_csv(F_DENGUE, encoding='utf-8-sig', sep=';', dtype=str,
                        keep_default_na=False, on_bad_lines='warn')

def norm_col(c):
    c = unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^0-9a-z]+', '_', c.strip().lower()).strip('_')

for df in (df_ipress, df_dengue):
    df.columns = [norm_col(c) for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.strip()

print(f"IPRESS: {len(df_ipress):,} filas x {df_ipress.shape[1]} columnas")
print(f"Dengue: {len(df_dengue):,} filas x {df_dengue.shape[1]} columnas")

# --- 2. Verificación: ninguna fila se perdió al leer ---
def contar_lineas(ruta, enc):
    with open(ruta, encoding=enc, errors='replace') as f:
        return sum(1 for _ in f) - 1
print(f"\nIPRESS -> líneas del archivo: {contar_lineas(F_IPRESS, 'latin-1'):,} "
      f"vs filas cargadas: {len(df_ipress):,}")
print(f"Dengue -> líneas del archivo: {contar_lineas(F_DENGUE, 'utf-8-sig'):,} "
      f"vs filas cargadas: {len(df_dengue):,}")

# --- 3. IPRESS: nulos efectivos (vacío + marcadores textuales) ---
marc = {'', 'NULL', 'ACTUALIZAR', 'NO TIENE', 'NO CUENTA CON TELEFONO'}
nulos_ip = df_ipress.apply(lambda s: s.str.upper().isin(marc).sum())
print("\n--- IPRESS: nulos efectivos por columna (>0) ---")
display(nulos_ip[nulos_ip > 0].sort_values(ascending=False)
        .to_frame('nulos').assign(pct=lambda d: (d.nulos/len(df_ipress)*100).round(1)))

# --- 4. IPRESS: duplicados por código único ---
dup_ip = df_ipress.duplicated('codigo_unico', keep=False)
print(f"\nIPRESS: filas con código repetido: {dup_ip.sum()} "
      f"({df_ipress.loc[dup_ip,'codigo_unico'].nunique()} códigos distintos)")

# --- 5. IPRESS: composición de 'categoria' = Sin Categoría (criterio de capacidad) ---
sin_cat = df_ipress[df_ipress['categoria'] == 'Sin Categoría']
con_intern_sin_cat = (sin_cat['tipo'] == 'ESTABLECIMIENTO DE SALUD CON INTERNAMIENTO').sum()
print(f"\nIPRESS: 'Sin Categoría' = {len(sin_cat):,} filas "
      f"({len(sin_cat)/len(df_ipress):.1%}), de las cuales {con_intern_sin_cat} "
      f"tienen internamiento (no se pueden excluir solo por categoría)")

# --- 6. Dengue: nulos ---
nulos_dg = (df_dengue == '').sum()
print("\n--- Dengue: nulos por columna (>0) ---")
display(nulos_dg[nulos_dg > 0].sort_values(ascending=False)
        .to_frame('nulos').assign(pct=lambda d: (d.nulos/len(df_dengue)*100).round(1)))

# --- 7. Dengue: duplicados exactos ---
dup_dg = df_dengue.duplicated().sum()
print(f"\nDengue: filas 100% idénticas: {dup_dg:,} ({dup_dg/len(df_dengue):.1%})")

# --- 8. Dengue: edades fuera de rango ---
edad = pd.to_numeric(df_dengue['edad'], errors='coerce')
anios_invalidos = ((df_dengue['tipo_edad'] == 'A') & (edad > 110)).sum()
print(f"Dengue: edad en años > 110: {anios_invalidos} filas")

# --- 9. Cobertura: casos en ubigeos sin ninguna IPRESS ---
ub_ip = set(df_ipress['ubigeo'])
sin_ipress = ~df_dengue['ubigeo'].isin(ub_ip)
print(f"\nCasos en ubigeos sin IPRESS: {sin_ipress.sum():,} "
      f"en {df_dengue.loc[sin_ipress,'ubigeo'].nunique()} ubigeos distintos")

print("\nDiagnóstico completo. Datos listos para la celda de transformación.")

IPRESS: 20,819 filas x 33 columnas
Dengue: 1,029,421 filas x 14 columnas

IPRESS -> líneas del archivo: 20,819 vs filas cargadas: 20,819
Dengue -> líneas del archivo: 1,029,421 vs filas cargadas: 1,029,421

--- IPRESS: nulos efectivos por columna (>0) ---


,nulos,pct
inspeccion,20819,100.0
situacion,20819,100.0
camas,20189,97.0
cota,13377,64.3
este,12863,61.8
norte,12863,61.8
unidad_ejecutora,12723,61.1
nro_doc_categorizacion,12291,59.0
red,11887,57.1
telefono,1817,8.7



IPRESS: filas con código repetido: 52 (26 códigos distintos)

IPRESS: 'Sin Categoría' = 5,815 filas (27.9%), de las cuales 162 tienen internamiento (no se pueden excluir solo por categoría)

--- Dengue: nulos por columna (>0) ---


,nulos,pct
localcod,148616,14.4
localidad,53909,5.2
diresa,23,0.0



Dengue: filas 100% idénticas: 152,091 (14.8%)
Dengue: edad en años > 110: 14 filas

Casos en ubigeos sin IPRESS: 2,140 en 16 ubigeos distintos

Diagnóstico completo. Datos listos para la celda de transformación.


In [2]:
# ==============================================================================
# Fase 3: TRANSFORMACIÓN Y LIMPIEZA
# Corrige valores, no elimina filas ni columnas. Cada cambio queda documentado
# en comentarios y en un resumen impreso al final (registro de auditoría manual;
# ==============================================================================
import pandas as pd
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

# --------------------------------------------------------------------
# IPRESS
# --------------------------------------------------------------------

# 1. Deduplicar por codigo_unico: de las 52 filas repetidas (26 códigos),
#    se conserva la que tiene coordenadas (norte/este) cuando difieren;
#    si son copias exactas, se conserva la primera.
df_ipress['_tiene_coord'] = df_ipress['norte'].ne('') & df_ipress['este'].ne('')
df_ipress = (df_ipress
             .sort_values('_tiene_coord', ascending=False)
             .drop_duplicates('codigo_unico', keep='first')
             .drop(columns='_tiene_coord')
             .reset_index(drop=True))
print(f"IPRESS: deduplicado por codigo_unico -> {len(df_ipress):,} filas "
      f"(se retiraron 26 filas repetidas, no información: eran el mismo establecimiento)")

# 2. norte/este = "0" y "0" no es una coordenada real (0°N,0°E cae en el
#    golfo de Guinea) -> se trata como dato faltante.
n_cero = ((df_ipress['norte'] == '0') & (df_ipress['este'] == '0')).sum()
df_ipress.loc[(df_ipress['norte'] == '0') & (df_ipress['este'] == '0'),
              ['norte', 'este']] = np.nan
print(f"IPRESS: {n_cero} coordenadas '0,0' corregidas a nulo (no eran ubicación real)")

# 3. Marcadores de texto que en realidad significan "sin dato".
MARCADORES = ['NULL', 'ACTUALIZAR', 'NO TIENE', 'NO CUENTA CON TELEFONO', '']
n_marc = df_ipress.isin(MARCADORES).sum().sum()
df_ipress = df_ipress.replace(MARCADORES, np.nan)
print(f"IPRESS: {n_marc:,} celdas con marcador de texto ('NULL', 'ACTUALIZAR', etc.) -> nulo")

# 4. Fecha centinela 01/01/1900 (o no parseable) -> nulo.
#    Se convierte la columna a tipo fecha real; lo que no es fecha válida queda NaT.
fecha = pd.to_datetime(df_ipress['inicio_de_actividad'], format='%d/%m/%Y', errors='coerce')
n_centinela = (df_ipress['inicio_de_actividad'] == '01/01/1900').sum()
fecha = fecha.mask(df_ipress['inicio_de_actividad'] == '01/01/1900')
df_ipress['inicio_de_actividad'] = fecha
print(f"IPRESS: {n_centinela} fechas '01/01/1900' (fecha desconocida) -> nulo; "
      f"columna convertida a tipo fecha")

# 5. clasificacion con el mismo valor repetido por error de digitación
#    (ej. "PUESTOS DE SALUD...,PUESTOS DE SALUD...") -> se queda con valores únicos.
def limpiar_clasificacion(valor):
    if pd.isna(valor):
        return valor
    partes = [p.strip() for p in valor.split(',')]
    unicas = list(dict.fromkeys(partes))          # quita repetidos, conserva orden
    return ','.join(unicas)

antes = df_ipress['clasificacion'].copy()
df_ipress['clasificacion'] = df_ipress['clasificacion'].apply(limpiar_clasificacion)
n_clasif = (antes != df_ipress['clasificacion']).sum()
print(f"IPRESS: {n_clasif} filas con 'clasificacion' repetida por digitación -> corregidas")

# 6. Renombrar: en el archivo fuente, 'norte' trae longitud y 'este' trae
#    latitud (etiquetas invertidas en el origen). Se corrige el nombre.
df_ipress = df_ipress.rename(columns={'norte': 'longitud', 'este': 'latitud'})
df_ipress[['longitud', 'latitud']] = df_ipress[['longitud', 'latitud']].apply(pd.to_numeric)

print(f"\nIPRESS listo: {len(df_ipress):,} filas, {df_ipress.shape[1]} columnas\n")

# --------------------------------------------------------------------
# DENGUE
# --------------------------------------------------------------------

edad_num = pd.to_numeric(df_dengue['edad'], errors='coerce')

# 7. Edad en años > 110 (14 filas, error de captura, ej. "71963641") -> nulo.
mask_invalida = (df_dengue['tipo_edad'] == 'A') & (edad_num > 110)
n_inv = mask_invalida.sum()
df_dengue.loc[mask_invalida, 'edad'] = np.nan
print(f"Dengue: {n_inv} filas con edad en años > 110 (dato de captura imposible) -> nulo")

# 8. Unificar unidad de edad: todo a años, sobrescribiendo 'edad' y
#    actualizando 'tipo_edad' a 'A' para que ambas columnas queden
#    coherentes entre sí (se pierde el detalle exacto en meses/días,
#    a cambio de no crear una columna adicional).
edad_num = pd.to_numeric(df_dengue['edad'], errors='coerce')   # recalcular tras el paso 7
es_mes = df_dengue['tipo_edad'] == 'M'
es_dia = df_dengue['tipo_edad'] == 'D'
n_mes, n_dia = es_mes.sum(), es_dia.sum()

df_dengue.loc[es_mes, 'edad'] = (edad_num[es_mes] / 12).round(2)
df_dengue.loc[es_dia, 'edad'] = (edad_num[es_dia] / 365).round(3)
df_dengue.loc[es_mes | es_dia, 'tipo_edad'] = 'A'
df_dengue['edad'] = pd.to_numeric(df_dengue['edad'], errors='coerce')

print(f"Dengue: {n_mes:,} filas con edad en meses y {n_dia:,} en días "
      f"convertidas a años; 'tipo_edad' actualizado a 'A' en esas filas "
      f"(el dato original en meses/días no se conserva aparte)")

# 9. provincia/distrito con el carácter corrupto '�' (Ñ perdida en el
#    archivo fuente) -> se reconstruyen usando el nombre correcto de
#    IPRESS para ese mismo ubigeo (localidad no se puede reparar así:
#    es texto libre sin catálogo, se deja como está).
mapa_prov = df_ipress.drop_duplicates('ubigeo').set_index('ubigeo')['provincia']
mapa_dist = df_ipress.drop_duplicates('ubigeo').set_index('ubigeo')['distrito']

corrupta_prov = df_dengue['provincia'].str.contains('\ufffd', na=False)
corrupta_dist = df_dengue['distrito'].str.contains('\ufffd', na=False)

df_dengue.loc[corrupta_prov, 'provincia'] = df_dengue.loc[corrupta_prov, 'ubigeo'].map(mapa_prov)
df_dengue.loc[corrupta_dist, 'distrito'] = df_dengue.loc[corrupta_dist, 'ubigeo'].map(mapa_dist)

print(f"Dengue: {corrupta_prov.sum():,} nombres de provincia y "
      f"{corrupta_dist.sum():,} de distrito reparados usando el catálogo IPRESS "
      f"(vía ubigeo); localidad no se repara, queda documentado como limitación")

print(f"\nDengue listo: {len(df_dengue):,} filas, {df_dengue.shape[1]} columnas")

IPRESS: deduplicado por codigo_unico -> 20,793 filas (se retiraron 26 filas repetidas, no información: eran el mismo establecimiento)
IPRESS: 3 coordenadas '0,0' corregidas a nulo (no eran ubicación real)
IPRESS: 140,797 celdas con marcador de texto ('NULL', 'ACTUALIZAR', etc.) -> nulo
IPRESS: 1117 fechas '01/01/1900' (fecha desconocida) -> nulo; columna convertida a tipo fecha
IPRESS: 32 filas con 'clasificacion' repetida por digitación -> corregidas

IPRESS listo: 20,793 filas, 33 columnas

Dengue: 14 filas con edad en años > 110 (dato de captura imposible) -> nulo
Dengue: 6,377 filas con edad en meses y 324 en días convertidas a años; 'tipo_edad' actualizado a 'A' en esas filas (el dato original en meses/días no se conserva aparte)
Dengue: 15,561 nombres de provincia y 30,222 de distrito reparados usando el catálogo IPRESS (vía ubigeo); localidad no se repara, queda documentado como limitación

Dengue listo: 1,029,421 filas, 14 columnas


In [3]:
# ==============================================================================
# Fase 4: EXPORTACIÓN DE DATASETS LIMPIOS
# ==============================================================================
from google.colab import files

print("--- EXPORTACIÓN DE DATASETS LIMPIOS ---\n")

# A. Exportar Catálogo IPRESS limpio
archivo_ipress = 'Establecimientos_Nacional.csv'
df_ipress.to_csv(archivo_ipress, index=False, encoding='utf-8-sig')
print(f"-> '{archivo_ipress}' generado ({df_ipress.shape[0]:,} registros, "
      f"{df_ipress.shape[1]} columnas)")

# B. Exportar Vigilancia Dengue limpio
archivo_dengue = 'Casos_Dengue_Nacional.csv'
df_dengue.to_csv(archivo_dengue, index=False, encoding='utf-8-sig')
print(f"-> '{archivo_dengue}' generado ({df_dengue.shape[0]:,} registros, "
      f"{df_dengue.shape[1]} columnas)")

print("\n" + "-" * 60)
print("Descargando los archivos limpios a tu computadora...")
files.download(archivo_ipress)
files.download(archivo_dengue)

--- EXPORTACIÓN DE DATASETS LIMPIOS ---

-> 'Establecimientos_Nacional.csv' generado (20,793 registros, 33 columnas)
-> 'Casos_Dengue_Nacional.csv' generado (1,029,421 registros, 14 columnas)

------------------------------------------------------------
Descargando los archivos limpios a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>